# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [12]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [8]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [14]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'careers page',
   'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [9]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
SulphurAI/Sulphur-2-base
Updated
5 days ago
•
627k
•
862
openbmb/MiniCPM-V-4.6
Updated
13 minutes ago
•
16.8k
•
509
Zyphra/ZAYA1-8B
Updated
2 days ago
•
131k
•
477
HiDream-ai/HiDream-O1-Image
Updated
1 day ago
•
9.86k
•
310
deepseek-ai/DeepSeek-V4-Pro
Updated
8 days ago
•
2.59M
•
3.93k
Browse 2M+ models
Spaces
Running
on
Zero
MCP
1.15k
Wan2.2 14B Fast Preview
🐌
1.15k
generate a video from an image with a text prompt
Running
154
The ultimate guide to RL environments: building and scaling them in the LLM era
📝
154
Building and scaling

In [10]:
brochure_system_prompt = """
You are an helpful and professional, sales and marketing assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [11]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nSulphurAI/Sulphur-2-base\nUpdated\n5 days ago\n•\n627k\n•\n862\nopenbmb/MiniCPM-V-4.6\nUpdated\n17 minutes ago\n•\n16.8k\n•\n509\nZyphra/ZAYA1-8B\nUpdated\n2 days ago\n•\n131k\n•\n478\nHiDream-ai/HiDream-O1-Image\nUpdated\n1 day ago\n•\n9.86k\n•\n310\ndeepseek-ai/DeepSeek-V4-Pro\nUpdated\n8 days ago\n•\n2.59M\n•\n3.93k\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nMCP\n1.15k\nWan2.2 14B

In [12]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [13]:
hindi_brochure_system_prompt = """
You are an helpful and professional, sales and marketing assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in hindi.
Include details of company culture, customers and careers/jobs if you have the information.
"""

def get_hindi_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in hindi language and in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [14]:
def create_brochure_hindi(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": hindi_brochure_system_prompt},
            {"role": "user", "content": get_hindi_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [15]:
create_brochure_hindi("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# हगिंगफेस (Hugging Face) कंपनी परिचय

## भविष्य का एआई समुदाय

हगिंगफेस एक अग्रणी कृत्रिम बुद्धिमत्ता (AI) और मशीन लर्निंग (ML) प्लेटफॉर्म है जो पूरे विश्व के डेवलपर्स, शोधकर्ताओं और डेटा वैज्ञानिकों को जोड़कर भविष्य के AI निर्माण में सहयोग करता है। यह प्लेटफॉर्म मशीन लर्निंग मॉडल, डेटासेट, और एप्लिकेशन के आदान-प्रदान व सहयोग के लिए बनाया गया है।

---

## हमारी विशेषताएँ

- **२ मिलियन से अधिक मॉडल:** हगिंगफेस पर लाखों खुले स्रोत मॉडल उपलब्ध हैं, जिनका उपयोग आप मशीन लर्निंग प्रोजेक्ट्स के लिए कर सकते हैं।
- **डेटासेट्स का विशाल संग्रह:** ५००,०००+ डेटासेट्स विभिन्न प्रकार के डेटा के साथ उपलब्ध हैं, जिन पर सहयोग और विश्लेषण किया जा सकता है।
- **Spaces:** उपयोगकर्ता अपने ML एप्लिकेशन और मॉडल के लिए स्पेस बना सकते हैं और उन्हें सार्वजनिक कर सकते हैं, जिससे वे तेजी से विकास कर सकें।
- **मॉडल सहयोग:** डेवलपर और शोधकर्ता खुलकर अपने मॉडल और डेटासेट साझा करते हैं जिससे मशीन लर्निंग कम्युनिटी तेजी से आगे बढ़ती है।
- **मल्टीमॉडल सपोर्ट:** टेक्स्ट, इमेज, वीडियो, ऑडियो और 3D जैसे अनेक डेटा प्रारूपों का समर्थन करता है, जिससे विविध क्षेत्रों में AI का प्रयोग संभव है।

---

## कंपनी संस्कृति

हगिंगफेस की संस्कृति सहयोग, नवाचार, और खुलापन पर आधारित है। कंपनी का मिशन हर व्यक्ति को मशीन लर्निंग की शक्ति तक पहुंचाना है। "एक कमिट में एक बेहतर मशीन लर्निंग" के दृष्टिकोण से टीम निरंतर खुला स्रोत (Open Source) मॉडल्स और टूल्स पर काम करती है। 

- **सहयोग केंद्रित माहौल:** कर्मचारियों को आपस में मिलकर काम करने और ज्ञान साझा करने के लिए प्रोत्साहित किया जाता है।
- **नवाचार को बढ़ावा:** नई तकनीकों और आईडिया के लिए खुलापन बना रहता है।
- **समाज सेवा:** AI के democratization (सर्वसुलभ बनाने) के लिए काम करता है ताकि AI टेक्नोलॉजी का लाभ सभी को मिल सके।

---

## ग्राहक और उपयोगकर्ता

हगिंगफेस की सेवाएं और प्लेटफॉर्म का उपयोग विभिन्न क्षेत्रों में किया जाता है, जैसे:

- AI रिसर्च और विकास संस्थान
- टेक कंपनियां और स्टार्टअप्स
- बड़े उद्यम जिनमें AI आधारित समाधान लागू करने होते हैं
- स्वतंत्र शोधकर्ता और डेवलपर्स जो AI आधारित एप्लिकेशन बनाना चाहते हैं
- शिक्षण संस्थान और विद्यार्थी

---

## करियर अवसर

हगिंगफेस टीम में शामिल होने वाले लोग मशीन लर्निंग, डेटा साइंस, सॉफ्टवेयर डेवलपमेंट, और AI रिसर्च के क्षेत्र में काम करने के इच्छुक होते हैं। कंपनी युवा और अनुभवी दोनों तरह के प्रतिभाओं को प्रोत्साहित करती है जो AI के भविष्य को आकार देना चाहते हैं।

- **टीम का आकार:** लगभग 185+ कर्मचारियों की एक सक्रिय टीम
- **रोजगार:** मशीन लर्निंग मॉडलिंग, डेटा इंजीनियरिंग, सॉफ्टवेयर विकास, और समुदाय प्रबंधन में नौकरियां उपलब्ध हैं।
- **सेवा-उन्मुख वार्ता:** टीम सभी के लिए समान अवसर और सहायक कार्य वातावरण प्रदान करती है।

---

## संपर्क और अधिक जानकारी

- आधिकारिक वेबसाइट: [huggingface.co](https://huggingface.co)
- सोशल मीडिया पर सक्रिय: ट्विटर और लिंक्डइन पर Hugging Face को फॉलो करें

---

हगिंगफेस के साथ जुड़ें और मशीन लर्निंग के इस खुला स्रोत और सहयोगी समुदाय का हिस्सा बनें।

  
**Hugging Face**  
*"The AI community building the future."*  

---

# संक्षेप में

हगिंगफेस मशीन लर्निंग कम्युनिटी के लिए एक खुला, सहयोगी और नवाचारी प्लेटफॉर्म है जहां मॉडल, डेटासेट्स, और AI एप्लिकेशन का निर्माण और साझा किया जाता है। कंपनी का उद्देश्य AI को अधिक सुलभ बनाना और इसके द्वारा विश्व की समस्याओं का समाधान करना है। यदि आप AI और मशीन लर्निंग के क्षेत्र में करियर बनाना चाहते हैं, या इस क्रांति का हिस्सा बनना चाहते हैं तो हगिंगफेस आपके लिए एक आदर्श स्थान है।

In [ ]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 13 relevant links


# Hugging Face Brochure

---

## About Hugging Face  
Hugging Face is the vibrant AI community and platform that is **building the future of machine learning**. Its ecosystem empowers machine learning practitioners — from researchers and developers to enterprises — to collaborate and innovate by sharing models, datasets, and applications with ease. Hugging Face is the go-to destination for anyone passionate about AI, offering resources and tools that accelerate ML development across all modalities: text, images, video, audio, and even 3D.

---

## What Hugging Face Offers  

### Collaborative Platform  
- Host and collaborate on **unlimited public models, datasets, and applications**  
- Share and discover from over **2 million machine learning models** and **500,000+ datasets**  
- Build and showcase your AI portfolio and profile directly on the platform  

### Diverse AI Modalities  
- Support for a wide range of data types including text, images, video, audio, and 3D  
- Explore and deploy state-of-the-art ML applications and Spaces (interactive ML apps)  

### Open Source & Ecosystem  
- Utilize the trusted HF Open Source stack to move faster in your AI projects  
- Engage with a large ecosystem of trending models like voice cloning in 600+ languages and image-to-3D generation  

### Enterprise & Team Plans  
- Dedicated plans for growing teams and organizations to streamline AI development and deployment  
- Advanced collaboration, private storage, inference quota, and enterprise-grade features to support scaling ML projects  

### Flexible Pricing  
- Free access to core platform features and community models  
- **PRO subscription** at $9/month offering enhanced features such as increased storage, inference credits, higher priority compute access, and personal blog publishing  
- Tailored pricing and support for teams and enterprise customers  

---

## Company Culture  
Hugging Face fosters a passionate, collaborative, and open-minded community focused on **building the future of AI together**. The culture is inclusive and driven by innovation, with a commitment to open source values and accessible AI technology empowering creators worldwide. The platform itself is a testament to community-driven development, supporting the sharing of cutting-edge research and practical ML deployments.

---

## Career Opportunities  
Hugging Face offers exciting career prospects for those eager to work at the **cutting edge of AI technology**. Join a team that is shaping the machine learning landscape by building collaborative tools and infrastructure used globally. Current openings span various domains including research, engineering, community engagement, and enterprise solutions.

Explore the latest job openings on their careers page and become part of a mission-driven company where your work has a direct impact on the AI community and industry.

---

## Customers & Community  
- Machine Learning Researchers and Developers looking for state-of-the-art models and datasets  
- AI startups and technology-driven enterprises implementing scalable AI solutions  
- Educators and students building AI portfolios and learning through hands-on projects  
- Enterprises leveraging team plans to deploy AI efficiently at scale and maintain security  

Hugging Face serves a broad spectrum of users from hobbyists and researchers to Fortune 500 companies, united by the goal of advancing machine learning responsibly and collaboratively.

---

## Get Started  
- Explore millions of models and datasets at [huggingface.co](https://huggingface.co)  
- Sign up for free to build, share, and collaborate on your AI projects  
- Upgrade to PRO or enterprise plans to unlock premium features and scale your AI initiatives  

---

**Hugging Face – The AI community building the future.**  
Join the revolution in machine learning collaboration today!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 17 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the AI community building the future of machine learning. It is a collaborative platform where ML enthusiasts, engineers, and researchers come together to create, share, and explore machine learning models, datasets, and applications. With over 2 million models and hundreds of thousands of datasets hosted, Hugging Face stands as the central hub of the global machine learning ecosystem.

---

## What Hugging Face Offers

- **Models**: Access, share, and contribute to over 2 million open-source models spanning multiple AI modalities including text, image, video, audio, and even 3D.
- **Datasets**: Browse and utilize from a vast collection of over 500,000 datasets to power your machine learning projects.
- **Spaces**: Host and explore machine learning demos and apps built by the community.
- **Collaboration Platform**: Host unlimited public projects and collaborate seamlessly with the global AI community.
- **Open Source Stack**: Move faster and innovate more by leveraging Hugging Face’s open-source tools and infrastructure.

---

## Community and Collaboration

Hugging Face fosters a vibrant, diverse, and inclusive community of ML practitioners. It empowers the next generation of machine learning engineers, scientists, and users to learn, experiment, and share their work openly. The platform enables users to:

- Build and showcase their ML portfolios.
- Collaborate globally with peers on open-source projects.
- Access trending, cutting-edge models and datasets updated regularly.
- Explore across all AI modalities and emerging areas like reinforcement learning environments and voice cloning across 600+ languages.

---

## Company Culture

Hugging Face embodies a culture centered on **openness, collaboration, and innovation**. The company values:

- Building an **open and accessible AI ecosystem**.
- Supporting a **community-first mindset** where sharing and collective progress drive the technology forward.
- Encouraging **experimentation and creativity** through accessible tools and open infrastructure.
- Empowering users from diverse backgrounds and expertise levels, fostering inclusivity in AI development.

---

## Customers & Users

Hugging Face is trusted by a broad spectrum of users including:

- Individual machine learning engineers and researchers.
- Academic and scientific communities.
- Enterprises integrating AI into their products and services.
- Developers building open-source and commercial AI applications.

Enterprises leverage Hugging Face’s scalable infrastructure and advanced tools to accelerate AI innovation and deployment.

---

## Careers & Opportunities

Hugging Face is continuously growing and seeking passionate individuals who share their vision of open, collaborative AI development. Roles often focus on:

- Machine Learning Engineering
- Research and Development
- Software Engineering
- Community and Developer Relations
- Product and Platform Development

Joining Hugging Face means being part of a mission-driven company that prioritizes curiosity, collaboration, and impact in the AI space.

---

## Brand Highlights

- Recognized for its iconic logo and bright brand colors (Yellow: #FFD21E, Orange: #FF9D00, Gray: #6B7280).
- Positioned as the home of open-source AI collaboration and innovation.
- A trusted name in the AI community for openness and developer empowerment.

---

## Get Involved

- Explore and contribute at the [Hugging Face Hub](https://huggingface.co)
- Share your models, datasets, or build AI applications on Spaces.
- Join the community to learn, collaborate, and propel the AI future.

---

**Hugging Face — The AI community building the future.**  
Create, discover, and collaborate on machine learning better.

---

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>